# ML-SAD — original exploratory notebook (outputs removed)

**Note.** This notebook is preserved for transparency as the original exploratory
analysis. Its cell *outputs have been stripped* because they printed raw,
identifying records (email addresses, etc.). It also reads the private raw JSON
files, which are **not** distributed.

For the maintained, de-identified, reproducible pipeline use the `scripts/` and
`src/ml_sad/` package, which run on the public `data/` tables. This notebook is
reference-only and is not guaranteed to run as-is.


In [ ]:
# pip install factor_analyzer

In [ ]:
# pip install statsmodels

In [ ]:
import json
import pandas as pd
from scipy.stats import ttest_ind
from scipy import stats
from factor_analyzer import FactorAnalyzer
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from factor_analyzer import (ConfirmatoryFactorAnalyzer, ModelSpecificationParser)
from statsmodels.multivariate.manova import MANOVA
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import kruskal
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
from sklearn.model_selection import KFold
from sklearn.model_selection import RepeatedKFold, RepeatedStratifiedKFold
from sklearn.model_selection import cross_val_score

In [ ]:
data = [json.loads(line) for line in open('fileV206_edited.json', 'r')]
print(len(data))

In [ ]:
print(data)

In [ ]:
data1 = pd.DataFrame.from_dict(data[-1], orient='index')
data1.head()

In [ ]:
df = pd.DataFrame()
for d in data:
    # Convert each dictionary into a single-row DataFrame
    temp_df = pd.DataFrame([d])
    df = pd.concat([df, temp_df], ignore_index=True)

print(df.head())



In [ ]:
df = df.drop(df[df['Age='].astype(int) > 35].index)
df = df.drop(df[df['Age='].astype(int) < 18].index)
print(len(df))

In [ ]:
mean = df['Age='].astype(float).mean(axis=0)
std = df['Age='].astype(float).std(axis=0)
print(mean)
print(std)

In [ ]:
female = df[df['Female='] == 1]
male = df[df['Female='] == 0]
print(len(female))
print(len(male))
print(len(female)/59)
print(len(male)/59)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# # Use a built-in matplotlib style for an attractive look.
# plt.style.use('seaborn-whitegrid')  

# Prepare the data: Convert the 'Age=' column to integers and drop missing values.
data = df['Age='].astype(int).dropna()

# Create a larger figure and axis.
fig, ax = plt.subplots(figsize=(6, 3))

# Plot the histogram with custom colors and edge definitions.
n, bins, patches = ax.hist(data, bins=18, color='mediumseagreen', edgecolor='darkgreen', alpha=0.8)

# Set the axes labels and title with increased font size and padding.
ax.set_xlabel('Age', fontsize=12, labelpad=10)
ax.set_ylabel('Frequency', fontsize=12, labelpad=10)
ax.set_title('Age Distribution', fontsize=12, pad=15)

# # Enhance grid aesthetics with dashed lines and controlled line width.
# ax.grid(True, linestyle='--', linewidth=0.7, alpha=0.7)

# Calculate the mean and add a vertical dashed line to mark it.
mean_age = data.mean()
ax.axvline(mean_age, color='navy', linestyle='dashed', linewidth=1, label=f'Mean: {mean_age:.1f}')

# Add a legend to identify the mean.
ax.legend(fontsize=10)

# Display the prettier plot.
plt.show()


In [ ]:
df.rename(columns = {'S1D1':'1S', 'S1D2':'1R', 'S1D3':'1A',
                     'S2D1':'2R', 'S2D2':'2S', 'S2D3':'2A',
                     'S3D1':'3S', 'S3D2':'3A', 'S3D3':'3R',
                     'S4D1':'4A', 'S4D2':'4R', 'S4D3':'4S',
                     'S5D1':'5A', 'S5D2':'5S', 'S5D3':'5R',
                     'S6D1':'6S', 'S6D2':'6R', 'S6D3':'6A',
                     'S7D1':'7A', 'S7D2':'7S', 'S7D3':'7R',
                     'S8D1':'8A', 'S8D2':'8S', 'S8D3':'8R',
                     'S9D1':'9R', 'S9D2':'9S', 'S9D3':'9A',
                     'S10D1':'10A', 'S10D2':'10R', 'S10D3':'10S'}, inplace = True)


In [ ]:
df_s = df.copy()
df2 = df.drop(['Age=', 'Consent=', 'Diagnised=', 'Email=', 'DateTime', 'Female=', 'Stress Level=', 'Version', 'description'] , axis=1)

In [ ]:
df.head()

avering the total scores:

In [ ]:
sum_S = df['1S'] + df['2S'] + df['3S'] + df['4S'] + df['5S'] + df['6S'] + df['7S'] + df['8S'] + df['9S'] + df['10S']
sum_A = df['1A'] + df['2A'] + df['3A'] + df['4A'] + df['5A'] + df['6A'] + df['7A'] + df['8A'] + df['9A'] + df['10A']
sum_R = df['1R'] + df['2R'] + df['3R'] + df['4R'] + df['5R'] + df['6R'] + df['7R'] + df['8R'] + df['9R'] + df['10R']

# sum_S = df['1S'] + df['2S'] + df['3S'] + df['4S'] + df['6S'] + df['7S'] + df['8S'] + df['9S']
# sum_A = df['1A'] + df['2A'] + df['3A'] + df['4A'] + df['6A'] + df['7A'] + df['8A'] + df['9A']
# sum_R = df['1R'] + df['2R'] + df['3R'] + df['4R'] + df['6R'] + df['7R'] + df['8R'] + df['9R']
df['sum_S'] = sum_S
df['sum_A'] = sum_A
df['sum_R'] = sum_R
m = 10 * 8
df['supression_sum'] = sum_S / m
df['avoidance_sum'] = sum_A / m
df['mean_maladaptive'] = (df['supression_sum'] + df['avoidance_sum'])/2*m
df['reappraisal_sum'] = sum_R / m

# df['supression_sum'] = sum_S / sum
# df['avoidance_sum'] = sum_A / sum
# df['mean_maladaptive'] = (df['supression_sum'] + df['avoidance_sum'])/2
# df['reappraisal_sum'] = sum_R / sum
# df.head()

Cronbach's alpha:



In [ ]:
####################
import pandas as pd

def cronbach_alpha(df_items):
    k = df_items.shape[1]
    # item variances (across respondents)
    item_vars = df_items.var(axis=0)
    # total score variance
    total_var = df_items.sum(axis=1).var()
    return (k / (k-1)) * (1 - item_vars.sum() / total_var)

# subscale dataframes
S_items = df2[[f"{i}S" for i in range(1,11)]]
A_items = df2[[f"{i}A" for i in range(1,11)]]
R_items = df2[[f"{i}R" for i in range(1,11)]]

alpha_S = cronbach_alpha(S_items)
alpha_A = cronbach_alpha(A_items)
alpha_R = cronbach_alpha(R_items)

# overall
all_items = pd.concat([S_items, A_items, R_items], axis=1)
alpha_total = cronbach_alpha(all_items)

print("α (Reappraisal):", alpha_R)
print("α (Suppression):", alpha_S)
print("α (Avoidance):", alpha_A)
print("α (Total):", alpha_total)


Calculating Spearman correlation between each question and total scores:

In [ ]:
for i in (1,2,3,4,5,6,7,8,9,10):
   cor_s = stats.spearmanr(df[str(i)+'S'], df['sum_S'])
   print(i ,'S', cor_s)
   print('\n')
   cor_a = stats.spearmanr(df[str(i)+'A'], df['sum_A'])
   print(i, 'A', cor_a)
   print('\n')
   cor_r = stats.spearmanr(df[str(i)+'R'], df['sum_R'])
   print(i, 'R', cor_r)
   print('\n')

In [ ]:
Stress_gp = df[(df['Diagnised='] == 1) & ((df['Stress Level='] == '5') | (df['Stress Level='] == '4'))]
Normal_gp = df[(df['Diagnised='] == 0) & ((df['Stress Level='] == '1') | (df['Stress Level='] == '2'))]
print(len(Stress_gp))
print(len(Normal_gp))

# Stress_gp_sr = df[(df['Diagnised='] == 0) & ((df['Stress Level='] == '5') | (df['Stress Level='] == '4'))]
# Normal_gp_sr = df[(df['Diagnised='] == 1) & ((df['Stress Level='] == '1') | (df['Stress Level='] == '2'))]
# print(len(Stress_gp_sr))
# print(len(Normal_gp_sr))

df.loc[(((df['Stress Level='] == '5') | (df['Stress Level='] == '4')) & (df['Diagnised='] == 1)), 'Stress'] = 1
df.loc[(((df['Stress Level='] == '2') | (df['Stress Level='] == '1')) & (df['Diagnised='] == 0)), 'Stress'] = 0

In [ ]:
print(len(df[df['Stress Level='] == '1']))
print(len(df[df['Stress Level='] == '3']))

In [ ]:
female = df[df['Female='] == 1]
male = df[df['Female='] == 0]
print(len(female))
print(len(male))

In [ ]:
Stress_gp_female = df[(df['Diagnised='] == 1) & ((df['Stress Level='] == '5') | (df['Stress Level='] == '4') & (df['Female='] == 1))]
Stress_gp_male = df[(df['Diagnised='] == 1) & ((df['Stress Level='] == '5') | (df['Stress Level='] == '4') & (df['Female='] == 0))]
Normal_gp_female = df[(df['Diagnised='] == 0) & ((df['Stress Level='] == '1') | (df['Stress Level='] == '2') & (df['Female='] == 1))]
Normal_gp_male = df[(df['Diagnised='] == 0) & ((df['Stress Level='] == '1') | (df['Stress Level='] == '2') & (df['Female='] == 0))]

In [ ]:
df.head()

Normality test:

In [ ]:
#######################
print(stats.shapiro(Stress_gp['reappraisal_sum']))
print(stats.shapiro(Normal_gp['reappraisal_sum']))

print(stats.shapiro(Stress_gp['supression_sum']))
print(stats.shapiro(Normal_gp['supression_sum']))

print(stats.shapiro(Stress_gp['avoidance_sum']))
print(stats.shapiro(Normal_gp['avoidance_sum']))

In [ ]:
####################
print(stats.shapiro(female['reappraisal_sum']))
print(stats.shapiro(male['reappraisal_sum']))

print(stats.shapiro(female['supression_sum']))
print(stats.shapiro(male['supression_sum']))

print(stats.shapiro(female['avoidance_sum']))
print(stats.shapiro(male['avoidance_sum']))

In [ ]:
# df['Stress Level='] = df['Stress Level='].astype(int)
df.rename(columns = {'Stress Level=':'st'}, inplace = True)
df.rename(columns = {'Diagnised=':'d'}, inplace = True)
df.rename(columns = {'Female=':'f'}, inplace = True)
df.rename(columns = {'Age=':'ag'}, inplace = True)

Measuring the validity: MANOVA and MANCOVA

In [ ]:
# maov = MANOVA.from_formula('supression_sum + avoidance_sum + reappraisal_sum ~ Stress', data=df)
# print(maov.mv_test())

In [ ]:
maov = MANOVA.from_formula('supression_sum + avoidance_sum + reappraisal_sum ~ st + f + d + ag', data=df)
print(maov.mv_test())

measuring the validity: t-test

In [ ]:
# print('reappraisal = ',ttest_ind(Stress_gp['reappraisal_sum'], Normal_gp['reappraisal_sum']))
# print('supression = ',ttest_ind(Stress_gp['supression_sum'], Normal_gp['supression_sum']))
# print('avoidance = ',ttest_ind(Stress_gp['avoidance_sum'], Normal_gp['avoidance_sum']))
# print('mean_maladaptive = ',ttest_ind(Stress_gp['mean_maladaptive'], Normal_gp['mean_maladaptive']))

Man Whitney non-parametric test:

In [ ]:
#######################
print('reappraisal = ',stats.mannwhitneyu(Stress_gp['reappraisal_sum'], Normal_gp['reappraisal_sum']))
print('supression = ',stats.mannwhitneyu(Stress_gp['supression_sum'], Normal_gp['supression_sum']))
print('avoidance = ',stats.mannwhitneyu(Stress_gp['avoidance_sum'], Normal_gp['avoidance_sum']))
print('mean_maladaptive = ',stats.mannwhitneyu(Stress_gp['mean_maladaptive'], Normal_gp['mean_maladaptive']))

In [ ]:
#######################
def rank_biserial(U, n1, n2):
    return (2 * U) / (n1 * n2) - 1


n1r = len(Stress_gp['reappraisal_sum'])
n2r = len(Normal_gp['reappraisal_sum'])
Ur = 572.5
eff_r = rank_biserial(Ur, n1r, n2r)
print("Reappraisal effect size", eff_r)

n1s = len(Stress_gp['supression_sum'])
n2s = len(Normal_gp['supression_sum'])
Us = 1938
eff_s = rank_biserial(Us, n1s, n2s)
print("Suppression effect size", eff_s)

n1a = len(Stress_gp['avoidance_sum'])
n2a = len(Normal_gp['avoidance_sum'])
Ua = 2258
eff_a = rank_biserial(Ua, n1a, n2a)
print("Avoidance effect size", eff_a)

In [ ]:
# print('reappraisal = ',ttest_ind(female['reappraisal_sum'], male['reappraisal_sum']))
# print('supression = ',ttest_ind(female['supression_sum'], male['supression_sum']))
# print('avoidance = ',ttest_ind(female['avoidance_sum'], male['avoidance_sum']))
# print('mean_maladaptive = ',ttest_ind(female['mean_maladaptive'], male['mean_maladaptive']))

In [ ]:
print('reappraisal = ',stats.mannwhitneyu(Stress_gp_female['reappraisal_sum'],Stress_gp_female['reappraisal_sum']))
print('supression = ',stats.mannwhitneyu(Stress_gp_female['supression_sum'], Stress_gp_female['supression_sum']))
print('avoidance = ',stats.mannwhitneyu(Stress_gp_female['avoidance_sum'], Stress_gp_female['avoidance_sum']))
print('\n')
print('reappraisal = ',stats.mannwhitneyu(Stress_gp_male['reappraisal_sum'],Stress_gp_male['reappraisal_sum']))
print('supression = ',stats.mannwhitneyu(Stress_gp_male['supression_sum'], Stress_gp_male['supression_sum']))
print('avoidance = ',stats.mannwhitneyu(Stress_gp_male['avoidance_sum'], Stress_gp_male['avoidance_sum']))
print('\n')
print('reappraisal = ',stats.mannwhitneyu(Normal_gp_female['reappraisal_sum'],Normal_gp_female['reappraisal_sum']))
print('supression = ',stats.mannwhitneyu(Normal_gp_female['supression_sum'], Normal_gp_female['supression_sum']))
print('avoidance = ',stats.mannwhitneyu(Normal_gp_female['avoidance_sum'], Normal_gp_female['avoidance_sum']))
print('\n')
print('reappraisal = ',stats.mannwhitneyu(Normal_gp_male['reappraisal_sum'],Normal_gp_male['reappraisal_sum']))
print('supression = ',stats.mannwhitneyu(Normal_gp_male['supression_sum'], Normal_gp_male['supression_sum']))
print('avoidance = ',stats.mannwhitneyu(Normal_gp_male['avoidance_sum'], Normal_gp_male['avoidance_sum']))

In [ ]:
#######################
print('reappraisal = ',stats.mannwhitneyu(female['reappraisal_sum'], male['reappraisal_sum']))
print('supression = ',stats.mannwhitneyu(female['supression_sum'], male['supression_sum']))
print('avoidance = ',stats.mannwhitneyu(female['avoidance_sum'], male['avoidance_sum']))
print('mean_maladaptive = ',stats.mannwhitneyu(female['mean_maladaptive'], male['mean_maladaptive']))

Visualization:

In [ ]:
#######################
labels = ['Supression', 'Avoidance', 'Reappraisal']
sup_stress = df['supression_sum'].loc[(df['Stress'] == 1)].mean()
sup_normal = df['supression_sum'].loc[(df['Stress'] == 0)].mean()

avd_stress = df['avoidance_sum'].loc[(df['Stress'] == 1)].mean()
avd_normal = df['avoidance_sum'].loc[(df['Stress'] == 0)].mean()

reap_stress = df['reappraisal_sum'].loc[(df['Stress'] == 1)].mean()
reap_normal = df['reappraisal_sum'].loc[(df['Stress'] == 0)].mean()

stress = [sup_stress, avd_stress, reap_stress]
normal = [sup_normal, avd_normal, reap_normal]

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots()
rects1 = ax.bar(x - width/2, stress, width, label='SAD', color='#F8AD7D', edgecolor='#F8AD7D', linewidth=1.2)
rects2 = ax.bar(x + width/2, normal, width, label='No SAD', color='mediumseagreen', edgecolor='mediumseagreen', linewidth=1.2)

ax.set_ylabel('Scores')
ax.set_title('Scores of Emotion Regulation Strategies')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()

fig.tight_layout()

plt.show()

In [ ]:
#######################
labels = ['Supression', 'Avoidance', 'Reappraisal']
sup_women = df['supression_sum'].loc[(df['f'] == 1)].mean()
sup_men = df['supression_sum'].loc[(df['f'] == 0)].mean()

avd_women = df['avoidance_sum'].loc[(df['f'] == 1)].mean()
avd_men = df['avoidance_sum'].loc[(df['f'] == 0)].mean()

reap_women = df['reappraisal_sum'].loc[(df['f'] == 1)].mean()
reap_men = df['reappraisal_sum'].loc[(df['f'] == 0)].mean()

women = [sup_women, avd_women, reap_women]
men = [sup_men, avd_men, reap_men]

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots()
rects1 = ax.bar(x - width/2, women, width, label='Women', color='#D1AEC4')
rects2 = ax.bar(x + width/2, men, width, label='Men', color='#729BBA')

ax.set_ylabel('Scores')
ax.set_title('Scores of Emotion Regulation Strategies among Men and Women')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()

fig.tight_layout()

plt.show()

Take Pearson correlation for the strategies:

In [ ]:
print(stats.pearsonr(df['supression_sum'], df['reappraisal_sum']))
print(stats.pearsonr(df['avoidance_sum'], df['reappraisal_sum']))
print(stats.pearsonr(df['avoidance_sum'], df['supression_sum']))

In [ ]:
df.to_csv(r'data.csv')

Confirmatory factor analysis:

In [ ]:
scaler = StandardScaler()
df3 = df2.copy()
df3 = pd.DataFrame(scaler.fit_transform(df3), columns = df3.columns)

In [ ]:
from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity
chi_square_value,p_value=calculate_bartlett_sphericity(df3)
chi_square_value, p_value

In [ ]:
from factor_analyzer.factor_analyzer import calculate_kmo
kmo_all,kmo_model=calculate_kmo(df3)
kmo_model

In [ ]:
fa = FactorAnalyzer(30, rotation=None)
fa.fit(df3)
FactorAnalyzer(bounds=(0.005, 1), impute='median', is_corr_matrix=False,
               method='minres', n_factors=3, rotation=None, rotation_kwargs={},
               use_smc=True)

In [ ]:
ev = fa.get_eigenvalues()

In [ ]:
plt.scatter(range(30),ev[0])
plt.plot(range(30),ev[0])
plt.title('Scree Plot')
plt.xlabel('Factors')
plt.ylabel('Eigenvalue')
plt.grid()
plt.show()

In [ ]:
# fa = FactorAnalyzer(3, rotation='oblimax', method='principal', use_smc=True)
# fa.fit(df2)

# FactorAnalyzer(impute='median', is_corr_matrix=False,
#                method='principal', n_factors=3, rotation='oblimax',
#                rotation_kwargs={}, use_smc=True)
fa = FactorAnalyzer(2, rotation='varimax', method='minres', use_smc=True)
fa.fit(df3)

FactorAnalyzer(impute='median', is_corr_matrix=False,
               method='principal', n_factors=2, rotation='varimax',
               rotation_kwargs={}, use_smc=True)

In [ ]:
fa.get_communalities()

In [ ]:
# loadings = pd.DataFrame(fa.loadings_, columns=['Factor 1', 'Factor 2', 'Factor 3'], index=df2.columns)
# print('Factor Loadings \n%s' %loadings)
loadings = pd.DataFrame(fa.loadings_, columns=['Factor 1', 'Factor 2'], index=df3.columns)
print('Factor Loadings \n%s' %loadings)

In [ ]:
from mpl_toolkits import mplot3d
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
fig = plt.figure()

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np

# Data
y = df['Stress']*5
x = df['supression_sum']
z = df['avoidance_sum']

# Create figure
fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111, projection='3d')

# Scatter plot with better aesthetics
scatter = ax.scatter(
    x, y, z,
    c=y,                # color points by 'Stress' values
    cmap='winter',       # more vibrant colormap
    s=50,                # slightly bigger point size
    alpha=0.8,           # transparency for depth feeling
    edgecolors='w'       # white edges for clarity
)

# Labels
ax.set_xlabel('Suppression', fontsize=10, labelpad=10)
ax.set_ylabel('Stress Level', fontsize=10, labelpad=10)
ax.set_zlabel('Avoidance', fontsize=10, labelpad=10)

# Customize ticks
ax.tick_params(axis='both', which='major', labelsize=10)
ax.tick_params(axis='z', labelsize=10)

# # Add color bar
cbar = plt.colorbar(scatter, ax=ax, shrink=0.5, aspect=10, pad=0.11)
cbar.set_label('Stress Level', fontsize=10)

# Title
ax.set_title('Stress Level vs Suppression and Avoidance', fontsize=12, pad=0.1)

# View Angle
ax.view_init(elev=18, azim=145)  # rotate for a more dynamic angle

# Grid and background
ax.grid(True)
# ax.set_facecolor('#f0f0f0')
fig.patch.set_facecolor('white')

# Tight layout
plt.tight_layout()
plt.show()


In [ ]:
# Calculate sums using df_s (for merging with SPIN Questionnaire) instead of df
sum_S = df_s['1S'] + df_s['2S'] + df_s['3S'] + df_s['4S'] + df_s['5S'] + df_s['6S'] + df_s['7S'] + df_s['8S'] + df_s['9S'] + df_s['10S']
sum_A = df_s['1A'] + df_s['2A'] + df_s['3A'] + df_s['4A'] + df_s['5A'] + df_s['6A'] + df_s['7A'] + df_s['8A'] + df_s['9A'] + df_s['10A']
sum_R = df_s['1R'] + df_s['2R'] + df_s['3R'] + df_s['4R'] + df_s['5R'] + df_s['6R'] + df_s['7R'] + df_s['8R'] + df_s['9R'] + df_s['10R']

# Add new columns to df_s
df_s['sum_S'] = sum_S
df_s['sum_A'] = sum_A
df_s['sum_R'] = sum_R

# Compute normalized and derived metrics
m = 10
df_s['supression_sum']    = df_s['sum_S'] / m
df_s['avoidance_sum']     = df_s['sum_A'] / m
df_s['mean_maladaptive']  = (df_s['supression_sum'] + df_s['avoidance_sum']) / 2 * m
df_s['reappraisal_sum']   = df_s['sum_R'] / m


In [ ]:
df_s.head()

In [ ]:
import json

filepath = 'socialanxietyusers1.json'
data = []
ids = []

with open(filepath, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        try:
            record = json.loads(line)
        except json.JSONDecodeError:
            
            continue
        # print(line)
        data.append(record)

data_dict = {i: rec for i, rec in enumerate(data)}
# print(data_dict)
print(len(data_dict))

spin_data = []
for i in range(len(data_dict)):
    rec = data_dict[i]
    # print(rec)

    if 'spinquestionnaire' in rec and 'questionnaireScore' in rec['spinquestionnaire']:
        # score = rec['spinquestionnaire']['questionnaireScore']
        spin_data.append(data_dict[i])

print(len(spin_data))

total_ques = []
severity_labels = []
severity_cat = []

for rec in data_dict.values():
    id = rec.get('id')
    spin = rec.get('spinquestionnaire')
    if not spin:
        continue

    # Fragen extrahieren (entweder als Dict oder aus Einzel-Keys)
    if isinstance(spin.get('questions'), dict):
        questions = spin['questions']
    else:
        questions = {
            f'question{i}': spin.get(f'question{i}', 0)
            for i in range(1, 18)
    }

    # Summe aller Antworten
    total = sum(int(v) for v in questions.values())

    if 0 <= total <= 22:
        label = 0
    elif 23 <= total <= 45:
        label = 5
    elif 46 <= total:
        label = 1
    else:
        label = "Out of range"

    severity_labels.append(total)
    severity_cat.append(label)

    ids.append(id)

    

print(severity_labels)

suppression = []
avoidance = []
reappraisal = []
label = []


for i in range(1, 129):
    spin = data_dict[i]
    suppression.append(spin["supression_sum"])
    avoidance.append(spin["avoidance_sum"])
    reappraisal.append(spin["reappraisal_sum"])
    label.append(spin["label"])

# print(severity_labels)
# print(suppression)
# print(avoidance)
# print(reappraisal)
# print(label)

data_new = {
    "severity_labels":    severity_labels,
    "severity_cat":    severity_cat,
    "suppression_spin": suppression,
    "avoidance_spin":   avoidance,
    "reappraisal_spin": reappraisal,
    "label":       label
}


df_spin = pd.DataFrame()
df_spin['id'] = ids
df_spin['severity_labels'] = 0
df_spin['suppression_sum_spin'] = 0
df_spin['avoidance_sum_spin'] = 0
df_spin['reappraisal_sum_spin'] = 0
df_spin['label'] = 0
df_spin['severity_cat'] = severity_cat
df_spin['severity_labels'] = severity_labels
df_spin['suppression_sum_spin'] = suppression
df_spin['avoidance_sum_spin'] = avoidance
df_spin['reappraisal_sum_spin'] = reappraisal
df_spin['label'] = label



In [ ]:
import pandas as pd

# 1. Filter df_s rows where 'Email=' appears in sf_spin['id']
df_s_matched = df_s[df_s['Email='].isin(df_spin['id'])].copy()

# 2. Filter sf_spin rows where 'id' appears in df_s['Email=']
df_spin_matched = df_spin[df_spin['id'].isin(df_s['Email='])].copy()

# 3. Merge the two matched sets on the common key
merged_df = pd.merge(
    df_s_matched,
    df_spin_matched,
    left_on='Email=',
    right_on='id',
    how='inner',
    suffixes=('_s', '_spin')
)

merged_df.head()


In [ ]:
import pandas as pd
import numpy as np

cols = [
    'supression_sum',
    'avoidance_sum',
    'reappraisal_sum',
    'severity_labels'
]

corr_matrix = merged_df[cols].corr()

print("Correlation matrix:")
print(np.round(corr_matrix, 3))



In [ ]:
len(merged_df)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. Encode severity labels as numeric codes
merged_df['severity_labels_code'] = (
    merged_df['severity_labels']
      .astype('category')
      .cat.codes
)
categories = merged_df['severity_labels'].astype('category').cat.categories

# 2. Compute the mean of each measure within each severity category
grouped = (
    merged_df
      .groupby('severity_labels_code')[[
          'supression_sum',
          'avoidance_sum',
          'reappraisal_sum'
      ]]
      .mean()
      .reset_index()
)

# 3. Single wide plot with white background
plt.style.use('default')  # back to the default white background

fig, ax = plt.subplots(figsize=(10, 5), facecolor='white')
ax.set_facecolor('white')

markers = ['o', 's', '^']
for col, mk in zip(['supression_sum', 'avoidance_sum', 'reappraisal_sum'], markers):
    ax.plot(
        grouped['severity_labels_code'],
        grouped[col],
        marker=mk,
        linewidth=1,
        markersize=4,
        label=col.replace('_', ' ').title()
    )

ax.set_xlabel('Severity n', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_xticks(grouped['severity_labels_code'])
ax.set_xticklabels(categories, rotation=45, ha='right', fontsize=10)
ax.set_title('Emotion Regulation Scores Vs. Severity Scores from SPIN Questionnaire', fontsize=14, pad=15)
ax.legend(title='Measure', fontsize=10, title_fontsize=11)
ax.grid(False)

plt.tight_layout()
plt.show()


In [ ]:
merged_df.loc[(((merged_df['Stress Level='] == '5') | (merged_df['Stress Level='] == '4')) & (merged_df['Diagnised='] == 1)), 'Stress'] = 1
# merged_df.loc[(merged_df['Stress Level='] == '3'),  'Stress'] = 1
merged_df.loc[(((merged_df['Stress Level='] == '2') | (merged_df['Stress Level='] == '1')) & (merged_df['Diagnised='] == 0)), 'Stress'] = 0
merged_df.head()


In [ ]:
import pandas as pd
from sklearn.metrics import confusion_matrix

# 1. load or build your merged_df, including:
#    - merged_df['severity_cat'] (0,1,2)
#    - merged_df['Stress']       (0,1)

# 2. choose a binary prediction rule
#    here: any severity ≥1 predicts “stressed”
merged_df = merged_df.dropna(subset=['Stress'])
y_true = merged_df['Stress']
print(len(merged_df))
y_pred = (merged_df['severity_cat'] <= 1).astype(int)
print(merged_df['severity_labels'].corr(merged_df['Stress'], method='spearman'))

# 3. get confusion matrix
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

# 4. calculate metrics
sensitivity = tp / (tp + fn)   # TP / (TP + FN)
specificity = tn / (tn + fp)   # TN / (TN + FP)

print("Confusion matrix:")
print(f" TN: {tn}   FP: {fp}")
print(f" FN: {fn}   TP: {tp}\n")

print(f"Sensitivity (TPR): {sensitivity:.3f}")
print(f"Specificity (TNR): {specificity:.3f}")


In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np

# Data
y = df['supression_sum']
z = df['reappraisal_sum']
x = df['avoidance_sum']

# Create figure
fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111, projection='3d')

# Scatter plot with better aesthetics
scatter = ax.scatter(
    x, y, z,
    c=y,                # color points by 'Stress' values
    cmap='winter',       # more vibrant colormap
    s=50,                # slightly bigger point size
    alpha=0.8,           # transparency for depth feeling
    edgecolors='w'       # white edges for clarity
)

# Labels
ax.set_zlabel('Reappraisal', fontsize=10, labelpad=10)
ax.set_ylabel('Suppression', fontsize=10, labelpad=10)
ax.set_xlabel('Avoidance', fontsize=10, labelpad=10)

# Customize ticks
ax.tick_params(axis='both', which='major', labelsize=10)
ax.tick_params(axis='z', labelsize=10)

# # Add color bar
cbar = plt.colorbar(scatter, ax=ax, shrink=0.5, aspect=10, pad=0.11)
cbar.set_label('Suppression', fontsize=10)

# Title
ax.set_title('Reappraisal vs Suppression and Avoidance', fontsize=12, pad=0.1)

# View Angle
ax.view_init(elev=30, azim=135)  # rotate for a more dynamic angle

# Grid and background
ax.grid(True)
# ax.set_facecolor('#f0f0f0')
fig.patch.set_facecolor('white')

# Tight layout
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn import metrics
from sklearn.model_selection import train_test_split

In [ ]:
data = df
data[(data['d'] == 1) & ((data['st'] == '5') | (data['st'] == '4')) | (data['d'] == 0) & ((data['st'] == '3') | (data['st'] == '2') | (data['st'] == '1'))]
data.loc[((data['st'] == '5') & (data['d'] == 1)), 'label'] = 0
data.loc[((data['st'] == '4') & (data['d'] == 1)), 'label'] = 0
# data.loc[((data['st'] == '3')), 'label'] = 1
data.loc[((data['st'] == '2') & (data['d'] == 0)), 'label'] = 1
data.loc[((data['st'] == '1') & (data['d'] == 0)), 'label'] = 1
# data1 = data[(data['label'] == 1) | (data['label'] == 0) | (data['label'] == 2)]
data1 = data[(data['label'] == 1) | (data['label'] == 0)]

data2 = data1.drop(['ag', 'Consent=', 'd', 'Email=', 'DateTime', 'f', 'st', 'Version', 'description', 'sum_S', 'sum_A', 'sum_R', 'supression_sum', 'avoidance_sum', 'reappraisal_sum', 'mean_maladaptive', 'Stress', 'label'] , axis=1)
data3 = data1.drop(['ag', 'Consent=', 'd', 'Email=', 'DateTime', 'f', 'st', 'Version', 'description', 'sum_S', 'sum_A', 'sum_R', 'supression_sum', 'avoidance_sum', 'reappraisal_sum', 'mean_maladaptive', 'Stress'], axis=1)
X = data2
# X=data1[['supression_sum', 'avoidance_sum']]
y=data3['label']
print(X.shape)
print(y.shape)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)
print(X_train.shape)
print(y_train.shape)
# X_train = X_train.fillna(X_train.mean(), inplace=True)
# X_test = X_test.fillna(X_test.mean(), inplace=True)
# print(X_train.shape)
# print(y_train.shape)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
knn_model = KNeighborsClassifier(n_neighbors=1)
knn_model.fit(X_train, y_train)
y_pred = knn_model.predict(X_test)
print(y_pred)

In [ ]:
#######################
print("Accuracy:",metrics.accuracy_score(y_test, y_pred))
precision = metrics.precision_score(y_test, y_pred)
recall = metrics.recall_score(y_test, y_pred)
F1 = 2 * (precision * recall) / (precision + recall)
# print("Precision Score : ", metrics.precision_score(y_test, y_pred))
# print("Recall Score : ", metrics.recall_score(y_test, y_pred))
# print("f1_score : ", F1)

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=1)
print(np.mean(cross_val_score(knn_model, X, y, scoring='accuracy', cv=cv, n_jobs=-1)))
print(np.mean(cross_val_score(knn_model, X, y, scoring='precision', cv=cv, n_jobs=-1)))
print(np.mean(cross_val_score(knn_model, X, y, scoring='recall', cv=cv, n_jobs=-1)))
print(np.mean(cross_val_score(knn_model, X, y, scoring='f1', cv=cv, n_jobs=-1)))
print(np.mean(cross_val_score(knn_model, X, y, scoring='roc_auc', cv=cv, n_jobs=-1)))

In [ ]:
from sklearn.linear_model import Perceptron
perceptron = Perceptron()
perceptron.fit(X_train, y_train)
y_pred = perceptron.predict(X_test)

In [ ]:
#######################
print("Accuracy:",metrics.accuracy_score(y_test, y_pred))
precision = metrics.precision_score(y_test, y_pred)
recall = metrics.recall_score(y_test, y_pred)
F1 = 2 * (precision * recall) / (precision + recall)
# print("Precision Score : ", metrics.precision_score(y_test, y_pred))
# print("Recall Score : ", metrics.recall_score(y_test, y_pred))
# print("f1_score : ", F1)

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=1)
print(np.mean(cross_val_score(perceptron, X, y, scoring='accuracy', cv=cv, n_jobs=-1)))
print(np.mean(cross_val_score(perceptron, X, y, scoring='precision', cv=cv, n_jobs=-1)))
print(np.mean(cross_val_score(perceptron, X, y, scoring='recall', cv=cv, n_jobs=-1)))
print(np.mean(cross_val_score(perceptron, X, y, scoring='f1', cv=cv, n_jobs=-1)))
print(np.mean(cross_val_score(perceptron, X, y, scoring='roc_auc', cv=cv, n_jobs=-1)))

In [ ]:
from sklearn.svm import SVC
svc = SVC()
svc.fit(X_train, y_train)
y_pred = svc.predict(X_test)

In [ ]:
#######################
print("Accuracy:",metrics.accuracy_score(y_test, y_pred))
precision = metrics.precision_score(y_test, y_pred)
recall = metrics.recall_score(y_test, y_pred)
F1 = 2 * (precision * recall) / (precision + recall)
# print("Precision Score : ", metrics.precision_score(y_test, y_pred))
# print("Recall Score : ", metrics.recall_score(y_test, y_pred))
# print("f1_score : ", F1)

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=1)
print(np.mean(cross_val_score(svc, X, y, scoring='accuracy', cv=cv, n_jobs=-1)))
print(np.mean(cross_val_score(svc, X, y, scoring='precision', cv=cv, n_jobs=-1)))
print(np.mean(cross_val_score(svc, X, y, scoring='recall', cv=cv, n_jobs=-1)))
print(np.mean(cross_val_score(svc, X, y, scoring='f1', cv=cv, n_jobs=-1)))
print(np.mean(cross_val_score(svc, X, y, scoring='roc_auc', cv=cv, n_jobs=-1)))

In [ ]:
model = RandomForestClassifier(n_estimators=100, class_weight='balanced_subsample')
# define evaluation procedure
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=1)
# evaluate model
scores = cross_val_score(model, X, y, scoring='roc_auc', cv=cv, n_jobs=-1)
# summarize performance
print('Mean ROC AUC: %.3f' % np.mean(scores))
model.fit(X_train,y_train)
y_pred=model.predict(X_test)
print(y_pred)

In [ ]:
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=1)
print(np.mean(cross_val_score(model, X, y, scoring='accuracy', cv=cv, n_jobs=-1)))
print(np.mean(cross_val_score(model, X, y, scoring='precision', cv=cv, n_jobs=-1)))
print(np.mean(cross_val_score(model, X, y, scoring='recall', cv=cv, n_jobs=-1)))
print(np.mean(cross_val_score(model, X, y, scoring='f1', cv=cv, n_jobs=-1)))
print(np.mean(cross_val_score(model, X, y, scoring='roc_auc', cv=cv, n_jobs=-1)))

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import Perceptron
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

# --- Assume X, y already defined ---

# Step 1: Identify numeric and categorical columns
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = [col for col in X.columns if col not in numeric_cols]

# Step 2: Define preprocessing
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ],
    remainder='drop'
)

# Step 3: Define models
model_configs = {
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'SVM': SVC(kernel='rbf', probability=True),
    'Perceptron': Perceptron(),
    'RandomForest': RandomForestClassifier(n_estimators=100, class_weight='balanced_subsample')
}

# Step 4: Cross-validation config
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=1)
scoring_metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

# Step 5: Evaluate each model
for name, model in model_configs.items():
    print(f"\n=== {name} ===")
    pipeline = Pipeline(steps=[
        ('preprocess', preprocessor),
        ('classifier', model)
    ])
    for metric in scoring_metrics:
        score = np.mean(cross_val_score(pipeline, X, y, scoring=metric, cv=cv, n_jobs=-1))
        print(f"{metric:<9}: {score:.3f}")


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import RepeatedStratifiedKFold, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import Perceptron
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

# --- Assume X (DataFrame) and y are already defined ---

# Identify numeric and categorical columns
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = [col for col in X.columns if col not in numeric_cols]

# Define preprocessing pipeline
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ],
    remainder='drop'
)

# Define models and parameter grids for tuning
model_configs = {
    'KNN': (
        KNeighborsClassifier(),
        {
            'classifier__n_neighbors': [3, 5, 7, 9],
            'classifier__weights': ['uniform', 'distance'],
            'classifier__p': [1, 2]
        }
    ),
    'SVM': (
        SVC(probability=True),
        {
            'classifier__C': [0.1, 1, 10],
            'classifier__kernel': ['linear', 'rbf'],
            'classifier__gamma': ['scale', 'auto']
        }
    ),
    'Perceptron': (
        Perceptron(),
        {
            'classifier__penalty': [None, 'l2', 'l1', 'elasticnet'],
            'classifier__alpha': [0.0001, 0.001, 0.01],
            'classifier__max_iter': [1000],
            'classifier__tol': [1e-3]
        }
    ),
    'RandomForest': (
        RandomForestClassifier(class_weight='balanced_subsample'),
        {
            'classifier__n_estimators': [100, 200, 500],
            'classifier__max_depth': [None, 5, 10],
            'classifier__min_samples_split': [2, 5, 10],
            'classifier__max_features': ['sqrt', 'log2', None]
        }
    )
}

# Cross-validation configuration
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
scoring_metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

# Evaluate each model with hyperparameter tuning
results = {}
for name, (model, param_grid) in model_configs.items():
    print(f"\n=== {name} ===")
    pipeline = Pipeline(steps=[
        ('preprocess', preprocessor),
        ('classifier', model)
    ])
    # Tune hyperparameters on F1 score
    grid = GridSearchCV(pipeline, param_grid, cv=cv, scoring='f1', n_jobs=-1)
    grid.fit(X, y)
    best_model = grid.best_estimator_
    print(f"Best parameters: {grid.best_params_}")
    # Evaluate tuned model across metrics
    metrics_result = {}
    for metric in scoring_metrics:
        score = np.mean(cross_val_score(best_model, X, y, scoring=metric, cv=cv, n_jobs=-1))
        metrics_result[metric] = score
        print(f"{metric:<9}: {score:.3f}")
    results[name] = metrics_result

# results dict now holds the tuned performance metrics for all models


In [ ]:
# 3. confusion matrix
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
print("Confusion matrix:")
print(f" TN: {tn}   FP: {fp}")
print(f" FN: {fn}   TP: {tp}\n")

# 4. sensitivity & specificity
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
print(f"Sensitivity (TPR): {sensitivity:.3f}")
print(f"Specificity (TNR): {specificity:.3f}")

In [ ]:
# model.score(X_train, y_train)

In [ ]:
model.score(X_test, y_test)

In [ ]:
print(metrics.classification_report(y_test, y_pred, labels=[0,1]))

In [ ]:
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score
from matplotlib import pyplot

In [ ]:
plt.figure()
fpr, tpr, thresholds = roc_curve(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred)
plt.plot(fpr, tpr, label='ROC curve (area = %0.2f)' % roc_auc)

plt.plot([0, 1], [0, 1], 'k--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver operating characteristic example')
plt.legend(loc="lower right")
plt.show()

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, RepeatedStratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import Perceptron
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

# import ace_tools as tools

# ------------------------------------------------------------------
# 0. Assume `df` already exists in the workspace. If not, raise error
# ------------------------------------------------------------------
try:
    df
except NameError as e:
    raise RuntimeError("DataFrame `df` is not defined in the current Python environment.") from e

# ------------------------------------------------------------------
# 1. Reproduce user’s filtering / labeling logic
# ------------------------------------------------------------------
data = df.copy()

# Apply the conditional selection exactly as provided
mask = ((data['d'] == 1) & (data['st'].isin(['5', '4']))) | ((data['d'] == 0) & (data['st'].isin(['3', '2', '1'])))
data = data[mask]

# Create label column
data.loc[(data['st'] == '5') & (data['d'] == 1), 'label'] = 0
data.loc[(data['st'] == '4') & (data['d'] == 1), 'label'] = 0
data.loc[(data['st'] == '2') & (data['d'] == 0), 'label'] = 1
data.loc[(data['st'] == '1') & (data['d'] == 0), 'label'] = 1

# Keep only label 0 or 1
data = data[data['label'].isin([0, 1])]

# ------------------------------------------------------------------
# 2. Feature / target separation
# ------------------------------------------------------------------
drop_cols = [
    'ag', 'Consent=', 'd', 'Email=', 'DateTime', 'f', 'st', 'Version',
    'description', 'sum_S', 'sum_A', 'sum_R', 'supression_sum',
    'avoidance_sum', 'reappraisal_sum', 'mean_maladaptive', 'Stress'
]
X = data.drop(drop_cols + ['label'], axis=1)
y = data['label'].astype(int)

# Identify numeric & categorical columns (simple heuristic)
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = [col for col in X.columns if col not in numeric_cols]

# ------------------------------------------------------------------
# 3. Preprocessing: impute missing & scale numeric; impute categorical (most_frequent)
# ------------------------------------------------------------------
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent'))
    # encoding skipped; if categorical cols exist, consider OneHotEncoder
])

preprocess = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ],
    remainder='drop'
)

# ------------------------------------------------------------------
# 4. Train / test split (30% test)
# ------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ------------------------------------------------------------------
# 5. Model configs
# ------------------------------------------------------------------
models = {
    'KNeighborsRegressor': {
        'pipeline': Pipeline([
            ('prep', preprocess),
            ('model', KNeighborsRegressor())
        ]),
        'param_grid': {
            'model__n_neighbors': [3, 5, 7, 9],
            'model__weights': ['uniform', 'distance']
        },
        'scoring': 'neg_mean_squared_error'  # will threshold later for classification metrics
    },
    'Perceptron': {
        'pipeline': Pipeline([
            ('prep', preprocess),
            ('model', Perceptron())
        ]),
        'param_grid': {
            'model__alpha': [0.0001, 0.001, 0.01],
            'model__max_iter': [1000, 2000]
        },
        'scoring': 'accuracy'
    },
    'SVC': {
        'pipeline': Pipeline([
            ('prep', preprocess),
            ('model', SVC(probability=True))
        ]),
        'param_grid': {
            'model__C': [0.1, 1, 10],
            'model__kernel': ['linear', 'rbf']
        },
        'scoring': 'roc_auc'
    },
    'RandomForestClassifier': {
        'pipeline': Pipeline([
            ('prep', preprocess),
            ('model', RandomForestClassifier(n_estimators=50))
        ]),
        'param_grid': {
            'model__n_estimators': [50, 100, 200],
            'model__max_depth': [None, 10, 20]
        },
        'scoring': 'roc_auc'
    }
}

outer_cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=42)
results = []

# ------------------------------------------------------------------
# 6. Nested CV, fitting, evaluation
# ------------------------------------------------------------------
for name, cfg in models.items():
    grid = GridSearchCV(
        estimator=cfg['pipeline'],
        param_grid=cfg['param_grid'],
        cv=RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=42),
        scoring=cfg['scoring'],
        n_jobs=-1
    )
    # Unbiased nested CV performance estimate
    nested_scores = cross_val_score(grid, X_train, y_train, cv=outer_cv, scoring=cfg['scoring'], n_jobs=-1)
    
    # Fit best model on full training set
    grid.fit(X_train, y_train)
    best_model = grid.best_estimator_
    
    # Predict on test set
    if name == 'KNeighborsRegressor':
        # Regressor outputs; convert to class label 0/1 by threshold 0.5
        y_pred_cont = best_model.predict(X_test)
        y_pred = (y_pred_cont >= 0.5).astype(int)
        # For ROC AUC we need probabilities; use min/max scaling of predictions
        y_proba = (y_pred_cont - y_pred_cont.min()) / (y_pred_cont.max() - y_pred_cont.min())
    else:
        y_pred = best_model.predict(X_test)
        if hasattr(best_model, "predict_proba"):
            y_proba = best_model.predict_proba(X_test)[:, 1]
        else:
            # For Perceptron without predict_proba, use decision_function
            decision = best_model.decision_function(X_test)
            y_proba = (decision - decision.min()) / (decision.max() - decision.min())
    
    # Compute metrics
    test_metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, y_proba)
    }
    
    results.append({
        'model': name,
        'best_params': grid.best_params_,
        'nested_cv_mean': np.mean(nested_scores),
        'nested_cv_std': np.std(nested_scores),
        'test_accuracy': test_metrics['accuracy'],
        'test_precision': test_metrics['precision'],
        'test_recall': test_metrics['recall'],
        'test_f1': test_metrics['f1'],
        'test_roc_auc': test_metrics['roc_auc']
    })

df_results = pd.DataFrame(results)
print("Model performance comparison", df_results)


## Diagnostic 2x2 confusion matrices for all selected classifiers (DTA meta-analysis request)

Reconstructs the 2x2 table (TP / FP / FN / TN) underlying the diagnostic-accuracy
results in Table 8, for **all four selected classifiers** on a single common hold-out
test set (test_size = 0.30 -> n = 32, matching the paper).

**Positive class = SAD** (ground truth = self-reported prior expert clinical diagnosis,
combined with the extreme-groups design: SAD = diagnosed + high/very-high anxiety;
non-SAD = undiagnosed + low/very-low anxiety). Note: the notebook internally codes
SAD = 0, so here the target is re-oriented to disease-positive (SAD = 1) as required
for sensitivity/specificity in a DTA meta-analysis. `random_state=49` reproduces the
published operating point (KNN: sensitivity 0.79, specificity 0.78).

In [ ]:
import numpy as np, pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import Perceptron
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

# --- Rebuild feature matrix and disease-positive target from df (extreme groups) ---
_feat = [f"{i}{s}" for i in range(1, 11) for s in ("S", "A", "R")]
_d = df.copy()
_d.loc[((_d["st"] == "5") & (_d["d"] == 1)), "label"] = 0   # SAD
_d.loc[((_d["st"] == "4") & (_d["d"] == 1)), "label"] = 0   # SAD
_d.loc[((_d["st"] == "2") & (_d["d"] == 0)), "label"] = 1   # non-SAD
_d.loc[((_d["st"] == "1") & (_d["d"] == 0)), "label"] = 1   # non-SAD
_d = _d[_d["label"].isin([0, 1])]

X_cm = _d[_feat].apply(pd.to_numeric, errors="coerce")
X_cm = X_cm.fillna(X_cm.median())
y_cm = (1 - _d["label"].astype(int)).astype(int)            # SAD = 1 (positive), non-SAD = 0

SEED = 49
X_tr, X_te, y_tr, y_te = train_test_split(X_cm, y_cm, test_size=0.30, random_state=SEED)
print(f"Common hold-out test set: n = {len(y_te)}  |  SAD(+) = {int(y_te.sum())}  non-SAD(-) = {int((y_te==0).sum())}\n")

classifiers = {
    "KNN (k=5)":     KNeighborsClassifier(n_neighbors=5),
    "SVM (RBF)":     SVC(random_state=SEED),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight="balanced_subsample", random_state=SEED),
    "Perceptron":    Perceptron(random_state=SEED),
}

rows = []
for name, clf in classifiers.items():
    clf.fit(X_tr, y_tr)
    y_hat = clf.predict(X_te)
    tn, fp, fn, tp = confusion_matrix(y_te, y_hat, labels=[0, 1]).ravel()   # positive = SAD = 1
    sens = tp / (tp + fn) if (tp + fn) else 0
    spec = tn / (tn + fp) if (tn + fp) else 0
    ppv  = tp / (tp + fp) if (tp + fp) else 0
    npv  = tn / (tn + fn) if (tn + fn) else 0
    acc  = (tp + tn) / (tp + tn + fp + fn)
    f1   = 2 * ppv * sens / (ppv + sens) if (ppv + sens) else 0
    lrp  = sens / (1 - spec) if spec < 1 else np.inf
    lrn  = (1 - sens) / spec if spec else np.inf
    dor  = lrp / lrn if lrn not in (0, np.inf) else np.inf
    youden = sens + spec - 1
    rows.append([name, tp, fp, fn, tn, round(sens,3), round(spec,3), round(ppv,3),
                 round(npv,3), round(acc,3), round(f1,3), round(lrp,2), round(lrn,2),
                 round(dor,2), round(youden,3)])
    print(f"{name}")
    print(f"                       Pred SAD(+)   Pred non-SAD(-)")
    print(f"   Actual SAD(+)          TP = {tp:>2}       FN = {fn:>2}")
    print(f"   Actual non-SAD(-)      FP = {fp:>2}       TN = {tn:>2}")
    print(f"   Sensitivity={sens:.3f}  Specificity={spec:.3f}  PPV={ppv:.3f}  NPV={npv:.3f}  "
          f"Accuracy={acc:.3f}  F1={f1:.3f}\n")

cm_summary = pd.DataFrame(rows, columns=["Classifier","TP","FP","FN","TN","Sensitivity","Specificity",
                                         "PPV","NPV","Accuracy","F1","LR+","LR-","DOR","Youden_J"])

# --- SPIN self-report comparator (Table 8, n = 33), if merged_df is available ---
try:
    _m = merged_df.dropna(subset=["Stress"]).copy()
    _yt = _m["Stress"].astype(int)                      # SAD = 1
    _yp = (_m["severity_cat"] <= 1).astype(int)
    tn, fp, fn, tp = confusion_matrix(_yt, _yp, labels=[0, 1]).ravel()
    cm_summary.loc[len(cm_summary)] = ["SPIN (self-report)", tp, fp, fn, tn,
        round(tp/(tp+fn),3), round(tn/(tn+fp),3), round(tp/(tp+fp),3), round(tn/(tn+fn),3),
        round((tp+tn)/(tp+tn+fp+fn),3), np.nan, np.nan, np.nan, np.nan,
        round(tp/(tp+fn)+tn/(tn+fp)-1,3)]
except NameError:
    print("(merged_df not in memory - run the SPIN cells to append the SPIN comparator row)")

pd.set_option("display.width", 200, "display.max_columns", 20)
print(cm_summary.to_string(index=False))
cm_summary.to_csv("ml_sad_confusion_matrices_by_classifier.csv", index=False)
print("\nSaved -> ml_sad_confusion_matrices_by_classifier.csv")
